# HW3: Regression and Classification

In this assignment you will preprocess the dataset and perform some basic regression and classification tasks. The learning outcome of this part is to know how one can pre-process a real-world dataset and perform a supervised learning task, and to understand some of the fundamental mechanisms behind these tasks.

##  Grading: 

Pass/Fail.

To Pass this HW you need to provide a complete and correct solution, passing all the tests.

## OUTLINE: 

Data pre-processing, regression task and classification task

1. Reading the files
2. Missing Values
3. Imputing categorical variables
4. Imputing numerical variables
5. Classification with Decision Tree, single split
6. Classification with Decision Tree, Cross validation
7. Interpretation of the results

## Important instructions:

Each function you make will be considered during the grading, so it is important to strictly follow input and output instructions stated in the skeleton code.

You must not change the names of the functions, since, if you do, the tests will fail.

Since this Homework is, in part, focused on having you implement creative solutions to impute missing data, if at any point of the homework you will use functions like fillna(), SimpleImputer(), IterativeImputer(), or packages like fancyimpute, missingpy, or similar, you will fail a test designed to spot these packages. Please, try to avoid circumventing this rule, since een if you manage to pass the homework, a similar task might be in the exam, and there you would be spotted for sure.

## Homework Scenario: Cleaning and Preparing Heart Disease Data

You have recently joined the **Data Science and Analytics Unit** at the *Global Health Institute (GHI)*, a non-profit organization focused on improving cardiovascular disease diagnosis through data-driven research.  

A junior data analyst from your team, **Franco**, sends you a message:

> “Hey, welcome to the team! We’re preparing a predictive model to help doctors identify patients at risk of heart disease using clinical data from several hospitals.  
>   
> We have two related datasets:
> - **Cleveland dataset** → this will be used for **training and validation**
> - **Hungary dataset** → this will serve as our **independent test set**
>
> Unfortunately, it looks like something went wrong during the data collection process: some values appear to have been **corrupted or lost**. Before we can train any classification model, we need to **inspect and clean the data**, handle **missing or inconsistent values**, and make sure it’s ready for modeling. I'm completely lost and I have a lot of other work, can you please help me with the cleaning and with creating some baselines classification models?”

Your task is to **analyze and clean the datasets** before **building a classifier** to predict whether a patient has heart disease.

In [3]:
# these are the libraries that you will need throughout the assignment
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline

from matplotlib.colors import ListedColormap

from HW3 import *


from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

RSEED = 8

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## *1.* Reading the files

### `Task: Read the datasets from the 'datasets' folder. Use the files called cleveland.csv and hungary.csv that you have downloaded in this archive.`

## Heart Disease Dataset — Column Descriptions

Someone has changed the names of some columns in the dataset, so make sure to use this description and refer to it for the "allowed" values.

Common sense is useful when evaluating some of the features: for example, in this dataset there is no column called weight, but, if there was one, since we are talking about humans and not ethereal beings, if a patient had a value of 0 in the weight column, this value could be due to a typo, or corrupted, and would need to be cleaned in some way.

| **Column** | **Description** |
|-------------|-----------------|
| **Age** | Age of the patient (in years). This dataset only includes adult patients. |
| **Sex** | Biological sex of the patient: `1 = male`, `0 = female`. |
| **ChestPainType** | Type of chest pain experienced: <br>• `1` = typical angina <br>• `2` = atypical angina <br>• `3` = non-anginal pain <br>• `4` = asymptomatic. |
| **RestBP** | Resting blood pressure (in mm Hg) measured on admission to the hospital. |
| **Chol** | Serum cholesterol level (in mg/dl). |
| **FBS** | Fasting blood sugar: `1` if fasting blood sugar > 120 mg/dl, otherwise `0`. |
| **RestECG** | Resting electrocardiographic results: <br>• `0` = normal <br>• `1` = ST-T wave abnormality <br>• `2` = showing probable or definite left ventricular hypertrophy. |
| **MaxHR** | Maximum heart rate achieved during the exercise test. |
| **ExAng** | Exercise-induced angina: `1` = yes, `0` = no. |
| **Oldpeak** | ST depression induced by exercise relative to rest (a measure of exercise-induced ischemia). |
| **Slope** | Slope of the peak exercise ST segment: <br>• `1` = upsloping <br>• `2` = flat <br>• `3` = downsloping. |
| **Ca** | Number of major vessels (0–3) colored by fluoroscopy (a measure of blood flow). |
| **Thal** | Thalassemia test result: <br>• `3` = normal <br>• `6` = fixed defect <br>• `7` = reversible defect. |
| **Num** | Diagnosis of heart disease (target variable): <br>`0` = no heart disease, `1–4` = presence of heart disease with increasing severity. |


In [7]:
# From the folder 'datasets', read the files cleveland.csv and hungary.csv into the dataframes cleveland and test, respectively.

cleveland = pd.read_csv("cleveland.csv")  # change this
test = pd.read_csv("hungary.csv")     # change this

In [8]:
# You can uncomment this to inspact the datasets
cleveland.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,53.0,1.0,3.0,130.0,246.0,1.0,2.0,173.0,0.0,0.0,1.0,3.0,3.0,0
1,54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0,3
2,222.0,1.0,4.0,125.0,249.0,1.0,2.0,144.0,1.0,1.2,2.0,1.0,3.0,1
3,58.0,1.0,4.0,100.0,234.0,0.0,0.0,156.0,0.0,0.1,1.0,1.0,7.0,2
4,51.0,0.0,4.0,130.0,305.0,0.0,0.0,142.0,1.0,1.2,2.0,0.0,7.0,2


In [9]:
test.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,47,0,2,140,257,0,0,135,0,1.0,1,?,?,0
1,52,1,4,112,342,0,1,96,1,1.0,2,?,?,1
2,41,0,2,125,184,0,-1,-1,0,0.0,?,?,?,0
3,58,1,4,135,222,0,0,100,0,0.0,?,?,?,0
4,54,0,2,140,309,?,1,140,0,0.0,?,?,?,0


In [6]:
# if you want to see information about the dataset, uncomment:
# cleveland.describe()

In [7]:
# if you want to see information about the dataset, uncomment:
# test.describe()

## *2.* Missing values

### `Task: use the function clean_data from the HW.py file to get a clean version of the cleveland and test dataframes.`

In [ ]:
# Write your code here
cleveland_cleaned, missing_values_cleveland = pd.DataFrame(), {} # change this
test_cleaned, missing_values_test = pd.DataFrame(), {} # change this

In [40]:
(cleveland['Ca']=='?').sum()

np.int64(4)

In [46]:
numerical_column = [ 'Age', 'RestBP','Chol','MaxHR','Oldpeak' ]
target = 'Num'
total_column = cleveland.columns.to_list()
categorical_column = []
for c in total_column:
    if c not in numerical_column and c != target:
        categorical_column.append(c)
categorical_column

['Sex', 'ChestPainType', 'FBS', 'RestECG', 'ExAng', 'Slope', 'Ca', 'Thal']

In [47]:
df_copy = cleveland.copy()

In [48]:
df_copy.replace('?',np.nan,inplace=True)

In [49]:
df_copy.isna().sum()

Age              0
Sex              0
ChestPainType    0
RestBP           0
Chol             0
FBS              0
RestECG          0
MaxHR            0
ExAng            0
Oldpeak          0
Slope            0
Ca               4
Thal             2
Num              0
dtype: int64

In [50]:
df_copy.dtypes

Age              float64
Sex              float64
ChestPainType    float64
RestBP           float64
Chol             float64
FBS              float64
RestECG          float64
MaxHR            float64
ExAng            float64
Oldpeak          float64
Slope            float64
Ca                object
Thal              object
Num                int64
dtype: object

In [51]:
# All are numeric column with values later on category column by its nature
for col in df_copy.columns:
    df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

In [52]:
# Categorical Values hai yeh
valid_values = {
        'Sex': [0, 1],
        'ChestPainType': [1, 2, 3, 4],
        'FBS': [0, 1],
        'RestECG': [0, 1, 2],
        'ExAng': [0, 1],
        'Slope': [1, 2, 3],
        'Ca': [0, 1, 2, 3],
        'Thal': [3, 6, 7],
        'Num': [0, 1, 2, 3, 4]
}
for col in numerical_column:
        if col in df_copy.columns:
            if col == 'Oldpeak':
            # Oldpeak can be 0, but not negative
                df_copy.loc[df_copy[col] < 0, col] = np.nan
            elif col == 'Age':
                # Age can't be <18 since adult patients bola hai or > 100
                df_copy.loc[(df_copy[col] < 18) | (df_copy[col] > 100), col] = np.nan
            elif col == 'Chol':
                # Cholesterol should be between 0 and 1000
                df_copy.loc[(df_copy[col] <=0) | (df_copy[col] >= 600), col] = np.nan 
            elif col == 'MaxHR':
                df_copy.loc[(df_copy[col] <=0) | (df_copy[col] >= 220), col] = np.nan  
            else:
                # For other continuous features, 0 or negative are invalid
                df_copy.loc[df_copy[col] <= 0, col] = np.nan
        # 
# Categorical change
for col in categorical_column:
    df_copy.loc[~df_copy[col].isin(valid_values[col]),col] = np.nan

In [56]:
missing_values_count= df_copy.isna().sum().to_dict()
missing_values_count
df_clean = df_copy

In [ ]:
def clean_data(df):
    
    
    """
    Task: Data Cleaning
    --------------------
    This function should take a pandas DataFrame as input and return a cleaned DataFrame.
    
    Instructions:
    - Handle missing values in categorical and numerical columns separately.
    - Handle incorrect data points (e.g., negative or null weight values) (I know that there is no weight column!).
    - Ensure that in the cleaned dataframe all the missing or incorrect values are encoded as NaN.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame to clean.

    Returns:
    pd.DataFrame: The cleaned DataFrame.
    missing_values_count (dict): A dictionary with the count of missing values per column after cleaning.
    """
    pass

## *3.* Imputing categorical variables

At the beginning of this file you can find the names of the columns and a description of their contents.

Determine which columns are categorical, and set their type to object.

Determine which columns are numerical, and set their type accordingly.

Do not include the target column in any of these lists!

In [57]:
categorical_columns =     categorical_column  # change this
numerical_columns = numerical_column      # change this

### ` Task: Split the cleveland_cleaned dataframe in a train and a validation set, using train_test_split from sklearn. `

The train set must be called train, the validation set must be called val. The size of the validation set must be 30% of the total size of the cleveland_cleaned dataframe. Use shuffle=True and stratify the split based on y_cleveland. Make sure that both train and val are dataframes, and that the columns have the correct names. Reset the indexes of all four the dataframes, using drop=True.

In [10]:
# Split the data into X and y, where X contains the features and y contains the target variable.
X_cleveland = pd.DataFrame()  # change this
y_cleveland = pd.DataFrame()  # change this

X_test = pd.DataFrame()       # change this
y_test = pd.DataFrame()       # change this

In [ ]:
X_train, X_val, y_train, y_val = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame() # change this

In [12]:
# # if you want to see information about the split dataset, uncomment:
# X_train.head(5)

In [13]:
# # if you want to see information about the split dataset, uncomment:
# X_val.head(5)

In [14]:
# To make the classification task easier, transform the target variable into a binary variable.
# If the target variable is 0, it should remain 0. If the target variable is more than 0, it should be transformed into 1.
y_train = pd.DataFrame()  # change this
y_val = pd.DataFrame()    # change this
y_test = pd.DataFrame()   # change this

### ` Task: use the impute_missing_categorical function from the HW.py file to impute the missing data from the categorical features in your dataframes. `

In [ ]:
# Write your code here
X_train_imputed_cat, X_val_imputed_cat, X_test_imputed_cat = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

## *4.* Imputing numerical variables

### ` Task: use the impute_missing_numeric function from the HW.py file to impute the missing data from the numeric features in your dataframes. `

In [ ]:
# Write your code here

X_train_imputed_num, X_val_imputed_num, X_test_imputed_num = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


### ` Task: use the merge_imputed function from the HW.py file to merge your imputed dataframes. `

In [ ]:
# Merge the train_imputed_cat and train_imputed_num datasets. Call the resulting dataset X_train_imputed.
# Merge the val_imputed_cat and val_imputed_num datasets. Call the resulting dataset X_val_imputed.
# Merge the test_imputed_cat and test_imputed_num datasets. Call the resulting dataset X_test_imputed.

# Write your code here
X_train_imputed = pd.DataFrame()
X_val_imputed = pd.DataFrame()
X_test_imputed = pd.DataFrame()


## *5.* Classification, using a single split 

### ` Use the function train_and_evaluate_single_split to produce classification results for your test set.`

In [18]:
# The hyperparameters for the tree should be:
# criterion: ['gini', 'entropy']
# max_depth: [3, 5, 10]
# The hyperparameters for the logistic regression should be:
# penalty: ['l1', 'l2']
# C: [0.1, 10]
# solver: ['liblinear']

# For each combination of hyperparameters, train a classification pipeline using your function.


from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

hyperparameters_tree = {} # change this
hyperparameters_logreg = {} # change this
performance_df = pd.DataFrame(columns=['params', 'F1 scores'])

# create a list from the grid of hyperparameters for each model, and create the models.

start = time.time() # DO NOT CHANGE/DELETE THIS LINE

for number in range(1, 11): # change this
    # call your function here, then concat the results to performance_df
    pass # remove this line

end = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with a single split: ', end - start) # DO NOT CHANGE/DELETE THIS LINE

Time elapsed to run the hyperparameter tuning with a single split:  3.4809112548828125e-05


In [ ]:
# Concatenate the train and validation datasets. Call the resulting datasets X and y.
X = pd.DataFrame()  # change this
y = pd.DataFrame()  # change this

# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

# Write your code here

## *6.* Classification with Decision Tree using Cross Validation

### ` Use the function train_and_evaluate_cross_validation to produce classification results for your test set.`

In [ ]:
# 1. Use the same hyperparameters from the previous task.
# 2. Create a dataframe to store the performance of the model with cross-validation, containing the columns 'params' and 'Average F1 scores'
# 3. You can reuse the parameter grids from the previous step.
# 4. Run your function for each combination of hyperparameters, using 5-fold cross-validation.
# 5. Concatenate the results to the dataframe created in step 2.



X = [[5,6], [10,11], [15,16], [20,21], [25,26], [30,31], [35,36], [40,41], [45,46], [50,51]]    # Delete this line
y = [0,1,0,1,0,1,0,1,0,1]                                                                       # Delete this line

X = pd.DataFrame(X)                                                                             # Delete this line
y = pd.DataFrame(y)                                                                             # Delete this line


# DO NOT FORGET TO DELETE THE PREVIOUS LINES. They are only to make the empty assignment run without errors,
# but they will destroy the data you need.

performance_df_cv = pd.DataFrame(columns=['params', 'F1 scores'])

start_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

# call your function here, then concat the results to performance_df_cv

end_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with Cross Validation: ', end_CV - start_CV) # DO NOT CHANGE/DELETE THIS LINE


Time elapsed to run the hyperparameter tuning with Cross Validation:  0.0020859241485595703


In [ ]:
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

## *7.* Interpretation of the results 

### ` Which model performs the best? `

Write your explanation here. Delete this text.

### ` Task: use the best model to produce predictions on the test set, then calculate the F1 score on the test set. What do you notice? `

### ` What is a possible explanation? `

from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

"""
HW.py

This file contains the skeleton for the functions you need to implement as part of your homework.
Each function corresponds to a specific task and includes instructions on what is expected.
"""

# Task 2: Data Cleaning
def clean_data(df):
    """
    Task: Data Cleaning
    --------------------
    This function should take a pandas DataFrame as input and return a cleaned DataFrame.
    
    Instructions:
    - Handle missing values in categorical and numerical columns separately.
    - Handle incorrect data points (e.g., negative or null weight values) (I know that there is no weight column!).
    - Ensure that in the cleaned dataframe all the missing or incorrect values are encoded as NaN.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame to clean.

    Returns:
    pd.DataFrame: The cleaned DataFrame.
    missing_values_count (dict): A dictionary with the count of missing values per column after cleaning.
    """
    pass


# Task 3: Categorical Features Imputation

def impute_missing_categorical(df_train, df_val, df_test, categorical_columns):
    """
    Task: Categorical Features Imputation
    --------------------------------------
    This function should handle missing values in categorical columns using appropriate techniques.
    We will skip scaling and encoding to keep things simple.
    
    Instructions:
    - Create subsets of the input DataFrames (train, validation, test) with only the categorical columns.
    - Use KNNImputer with k=5 and weights set to 'distance' to fill missing values in the categorical columns.
    - Ensure that the imputed values are approximated to the nearest value in the original dataset for each column, to avoid artifacts like decimal values.
    - If any "new value" is equidistant from two original values, choose the smaller one.
    - Add the column names to the resulting DataFrames after imputation.
    - The imputed dataframes should only contain the categorical columns.

    Parameters:
    df_train (pd.DataFrame): The training DataFrame.
    df_val (pd.DataFrame): The validation DataFrame.
    df_test (pd.DataFrame): The test DataFrame.
    categorical_columns (list): A list of column names corresponding to categorical features.

    Returns:
    pd.DataFrame: The training DataFrame with imputed categorical features.
    pd.DataFrame: The validation DataFrame with imputed categorical features.
    pd.DataFrame: The test DataFrame with imputed categorical features.
    """

    pass


# Task 4: Numerical Features Imputation
def impute_numerical_features(df_train, df_val, df_test, numerical_columns):
    """
    Task: Numerical Features Imputation
    ------------------------------------
    This function should handle missing values in numerical columns using appropriate techniques.
    Again, we will skip scaling to keep things simple.
    
    Instructions:
    - The function should take three datasets as input: df_train, df_val, and df_test.
    - The function should return three datasets as output: train_imputed_lasso, val_imputed_lasso, and test_imputed_lasso.
    - Use LassoRegressor in an iterative fashion to impute missing values in numerical columns.
    - Ensure that the imputation process is consistent and does not use any other imputer.
    - Follow these steps:
        1. Create a subset of the train dataset with only the numerical columns. Call this subset train_num.
        2. Create a subset of the val dataset with only the numerical columns. Call this subset val_num.
        3. Create a subset of the test dataset with only the numerical columns. Call this subset test_num.
        4a. Create a subset of train_num containing the rows with missing values. Call this subset train_num_missing.
        4b. Create a subset of train_num containing the rows without missing values. Call this subset train_num_not_missing.
        5a. Create a subset of val_num containing the rows with missing values. Call this subset val_num_missing.
        5b. Create a subset of val_num containing the rows without missing values. Call this subset val_num_not_missing.
        6a. Create a subset of test_num containing the rows with missing values. Call this subset test_num_missing.
        6b. Create a subset of test_num containing the rows without missing values. Call this subset test_num_not_missing.
        7a. Train a Lasso regression model on the correct subset (I am not telling you which one it is).
        7b. Using a Lasso regression, "predict" the missing values in the subsets that have missing values. Only predict the values in the column with the fewest missing values.
        8. Repeat steps 4-7 until all the missing values are imputed.
        9. Save the results in train_num_imputed_lasso, val_num_imputed_lasso, and test_num_imputed_lasso.
        10. Concatenate the imputed subsets with the subsets that did not contain missing values.
        11. Save the resulting datasets in train_imputed_lasso, val_imputed_lasso, and test_imputed_lasso.
        12. Ensure that the order of the rows in the final datasets matches the order in the original datasets.

    Parameters:
    df_train (pd.DataFrame): The training DataFrame.
    df_val (pd.DataFrame): The validation DataFrame.
    df_test (pd.DataFrame): The test DataFrame.
    numerical_columns (list): A list of column names corresponding to numerical features.

    Returns:
    pd.DataFrame: The training DataFrame with imputed numerical features.
    pd.DataFrame: The validation DataFrame with imputed numerical features.
    pd.DataFrame: The test DataFrame with imputed numerical features.
    """
    pass

def merge_imputed(df_cat, df_num):
    """
    Task: Merge Imputed DataFrames
    -------------------------------
    This function should merge the imputed categorical and numerical DataFrames.
    
    Instructions:
    - Merge the imputed categorical and numerical DataFrames on their indexes.
    - Ensure that the resulting DataFrame contains all columns from both input DataFrames.
    
    Parameters:
    df_cat (pd.DataFrame): The DataFrame with imputed categorical features.
    df_num (pd.DataFrame): The DataFrame with imputed numerical features.

    Returns:
    pd.DataFrame: The merged DataFrame containing both categorical and numerical features.
    """
    pass


# Task 5: Classification Using a Single Split
def train_and_evaluate_single_split(X_train, X_val, y_train, y_val, cat_cols, num_cols, model, hp):
    """
    Task: Classification Using a Single Split
    ------------------------------------------
    This function should train a classification pipeline on the training set and evaluate it on the validation set, using the provided parameters.

    Instructions:
    - Create a classification pipeline. It should include:
        - A OneHotEncoder for categorical features (handle_unknown='ignore').
        - A StandardScaler for numerical features.
        - The provided classification model.
    - Use ColumnTransformer to apply the appropriate transformations to categorical and numerical features.
    - Set the model parameters using the provided parameters dictionary.
    - Train the model using the training data (X_train, y_train).
    - Evaluate the model on the validation data (X_val, y_val) using F1 score.
    - Return the evaluation results (F1 score) for the given parameters combination.
    
    Parameters:
    X_train (pd.DataFrame): The training feature set.
    X_val (pd.DataFrame): The validation feature set.
    y_train (pd.Series): The training labels.
    y_val (pd.Series): The validation labels.
    model: The classification model to train.
    hp (dict): A dictionary of hyperparameters to set for the model.

    Returns:
    dict: A dictionary containing two keys: 'params' (training parameters) and 'F1 scores' (F1 score). Each key should have the correct value.
    """
    pass


# Task 6: Classification Using Cross-Validation
def train_and_evaluate_cross_validation(X, y, model, cat_cols, num_cols, hp, cv):
    """
    Task: Classification Using Cross-Validation
    --------------------------------------------
    This function should train and evaluate a classification model using cross-validation.
    
    Instructions:
    - Use cross-validation to train and evaluate the model, with shuffle set to True and using the specified number of folds (cv).
    - For each fold, create a classification pipeline similar to the one in Task 5.
    - Evaluate the model on each fold using F1 score.
    - Return the average F1 score across all folds for each parameter combination.
    - Ensure that the cross-validation process is reproducible (use random_state = 8).
    
    Parameters:
    X (pd.DataFrame): The feature set.
    y (pd.Series): The labels.
    model: The classification model to train.
    hp (dict): A dictionary of hyperparameters to set for the model.
    cv (int): The number of cross-validation folds.

    Returns:
    dict: A dictionary containing two keys: 'params' (training parameters) and 'Average F1 scores' (F1 score). Each key should have the correct value.
    """
    pass